# Skin Cancer Detection - Model Training Notebook

This notebook trains **6 models** for skin cancer classification on HAM10000:

| # | Model | Type |
|---|---|---|
| 1 | Sequential CNN | From scratch |
| 2 | Custom ResNet | From scratch |
| 3 | EfficientNetB0 | Transfer learning |
| 4 | ResNet50 | Transfer learning |
| 5 | DenseNet121 | Transfer learning |
| 6 | ViT | Transfer learning |

Plus **class imbalance experiment** (EfficientNet + class weights).

**Runtime**: ~3.5 hours total on T4 GPU.

## 1. GPU Check

In [ ]:
!nvidia-smi
import tensorflow as tf
print(f"\nTensorFlow: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs: {gpus}")
if not gpus:
    print("\nNO GPU! Go to: Runtime > Change runtime type > T4 GPU > Save")
else:
    print("\nGPU ready!")

## 2. Clone Repository

In [ ]:
import os
if not os.path.exists('/content/SkinCancer'):
    !git clone https://github.com/Ashutosh-Repos/SkinCancer.git /content/SkinCancer
else:
    !cd /content/SkinCancer && git pull
%cd /content/SkinCancer
!ls src/

## 3. Install Dependencies

In [ ]:
!pip install flask flask-cors python-dotenv tqdm tensorflow-hub -q
import sys
sys.path.insert(0, '/content/SkinCancer/src')
from config import MODEL_CONFIG, TRAINING_CONFIG, TRANSFER_LEARNING_MODELS
print(f"Models: {list(MODEL_CONFIG.keys())}")
print(f"Test split: {TRAINING_CONFIG['test_split']}")

## 4. Download Dataset (upload kaggle.json when prompted)

In [ ]:
import os
if not os.path.exists('data/images') or len(os.listdir('data/images')) < 10000:
    print("Upload your kaggle.json file:")
    from google.colab import files
    uploaded = files.upload()
    !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !python scripts/download_dataset.py
else:
    print("Dataset already exists!")
print(f"Images: {len([f for f in os.listdir('data/images') if f.endswith('.jpg')])}")

## 5. Mount Google Drive & Create Directories

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/SkinCancer'
for d in ['models/checkpoints', 'logs', 'results', 'results/gradcam']:
    os.makedirs(f'{DRIVE_DIR}/{d}', exist_ok=True)
    os.makedirs(d, exist_ok=True)
print(f"Drive: {DRIVE_DIR}")

---
## Model Training
---

### Model 1: Sequential CNN (~15 min)

In [ ]:
!cd /content/SkinCancer && python src/train.py --model sequential --epochs 30 --batch-size 64

### Model 2: Custom ResNet (~15 min)

In [ ]:
!cd /content/SkinCancer && python src/train.py --model resnet --epochs 16 --batch-size 64

### Model 3: EfficientNetB0 (~35 min)

In [ ]:
!cd /content/SkinCancer && python src/train.py --model efficientnet --batch-size 32

### Model 4: ResNet50 (~30 min)

In [ ]:
!cd /content/SkinCancer && python src/train.py --model resnet50 --batch-size 32

### Model 5: DenseNet121 (~30 min)

In [ ]:
!cd /content/SkinCancer && python src/train.py --model densenet --batch-size 32

### Model 6: Vision Transformer (~45 min)

In [ ]:
!cd /content/SkinCancer && python src/train.py --model vit --batch-size 16

---
## Class Imbalance Experiment

**IMPORTANT**: This cell first evaluates the base EfficientNet, then backs it up, then trains with class weights.

---

In [ ]:
import shutil

# Step 1: Evaluate base EfficientNet BEFORE overwriting
print("=" * 60)
print("Evaluating base EfficientNet (before class weights)")
print("=" * 60)
!cd /content/SkinCancer && python src/evaluate.py --model models/checkpoints/efficientnet_best.h5

# Step 2: Backup checkpoint + metadata + results
for ext in ['.h5', '_metadata.json']:
    src = f'models/checkpoints/efficientnet_best{ext}'
    dst = f'models/checkpoints/efficientnet_no_cw_best{ext}'
    if os.path.exists(src):
        shutil.copy(src, dst)

if os.path.exists('results/efficientnet_best'):
    shutil.copytree('results/efficientnet_best', 'results/efficientnet_no_cw_best', dirs_exist_ok=True)

print("\nBase EfficientNet backed up.")

# Step 3: Train with class weights (overwrites efficientnet_best.h5)
print("\n" + "=" * 60)
print("Training EfficientNet + Class Weights")
print("=" * 60)
!cd /content/SkinCancer && python src/train.py --model efficientnet --batch-size 32 --class-weights

---
## Evaluate All Models
---

In [ ]:
import glob

# Find all checkpoints (including backed-up no_cw version)
model_files = sorted(glob.glob('models/checkpoints/*_best.h5'))
print(f"Found {len(model_files)} models:")
for f in model_files:
    mb = os.path.getsize(f) / (1024*1024)
    print(f"  {os.path.basename(f):45s} ({mb:.1f} MB)")

# Evaluate each (skips already-evaluated base efficientnet since results exist)
for mf in model_files:
    name = os.path.basename(mf)
    results_dir = f"results/{name.replace('.h5', '')}"
    if os.path.exists(f"{results_dir}/evaluation_metrics.json"):
        print(f"\nSkipping {name} (already evaluated)")
        continue
    print(f"\n{'='*60}")
    print(f"Evaluating: {name}")
    print(f"{'='*60}")
    !cd /content/SkinCancer && python src/evaluate.py --model {mf}

## Model Comparison Table

In [ ]:
import json, pandas as pd

results = []
for mf in sorted(glob.glob('results/*/evaluation_metrics.json')):
    name = mf.split('/')[-2]
    with open(mf) as f:
        m = json.load(f)
    results.append({
        'Model': name,
        'Accuracy': f"{m['accuracy']*100:.2f}%",
        'F1 (Macro)': f"{m['f1_macro']*100:.2f}%",
        'F1 (Weighted)': f"{m['f1_weighted']*100:.2f}%",
        'Precision': f"{m['precision_macro']*100:.2f}%",
        'Recall': f"{m['recall_macro']*100:.2f}%",
    })

if results:
    df = pd.DataFrame(results)
    display(df)
    df.to_csv('results/model_comparison.csv', index=False)
    print("\nSaved to results/model_comparison.csv")
else:
    print("No results found.")

## Display Training Curves & Confusion Matrices

In [ ]:
from IPython.display import Image as IPImage, display

print("TRAINING CURVES")
for p in sorted(glob.glob('logs/*/training_history.png')):
    print(f"\n--- {p.split('/')[-2]} ---")
    display(IPImage(filename=p, width=800))

print("\nCONFUSION MATRICES")
for p in sorted(glob.glob('results/*/confusion_matrix.png')):
    print(f"\n--- {p.split('/')[-2]} ---")
    display(IPImage(filename=p, width=800))

## Save to Google Drive

In [ ]:
DRIVE_DIR = '/content/drive/MyDrive/SkinCancer'
!cp -r models/checkpoints/ {DRIVE_DIR}/models/checkpoints/
!cp -r logs/ {DRIVE_DIR}/logs/
!cp -r results/ {DRIVE_DIR}/results/
!cp data/norm_stats.json {DRIVE_DIR}/ 2>/dev/null || true
print("All outputs saved to Google Drive!")
!find {DRIVE_DIR} -name '*.h5' -o -name '*.json' -o -name '*.png' -o -name '*.csv' | head -30

## Download Best Model

In [ ]:
from google.colab import files
for f in ['models/checkpoints/efficientnet_best.h5',
          'models/checkpoints/efficientnet_best_metadata.json',
          'data/norm_stats.json',
          'results/model_comparison.csv']:
    if os.path.exists(f):
        files.download(f)
        print(f"Downloaded: {f}")
print("\nUsage on MacBook:")
print("  python src/inference.py --model models/checkpoints/efficientnet_best.h5 --image <image.jpg>")